In [1]:
import os
import sys

ROOT_PATH = '../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [ ]:
from types import SimpleNamespace
import gc
import csv

import torch
from peft import LoraConfig, get_peft_model
from ultralytics.utils.loss import v8SegmentationLoss, E2ELoss
from torch.optim import AdamW
from torch.nn import L1Loss
from torch.amp import autocast, GradScaler

from datasets.rescuenet_dataset import RescueNetDataset, collate_fn
from utils.augmentations import get_augmentation_pipeline
from model.TriheadSegmentationModel import TriheadSegmentationModel
from model.UncertaintyLossWeighting import UncertaintyLossWeighting
from utils.training import compute_normal_loss, EarlyStoppingAndCheckpointing
from custom_types.training import AblationStudyType
from utils.runpod import end_session

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device('cpu')
print(device)

cpu


In [3]:
MODEL_WEIGHT_DIR = 'model/weights'

In [4]:
def create_peft_model(model: TriheadSegmentationModel):
  valid_target_modules = []
  for name, module in model.yolo.model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
      if module.groups == 1:
        valid_target_modules.append(name)
  
  lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=valid_target_modules,
    modules_to_save=["model.23.*"],
    bias="none",
  )
  peft_backbone = get_peft_model(model.yolo.model, peft_config=lora_config).to(device)
  peft_backbone.print_trainable_parameters()
  peft_backbone.to(device)
  
  model.yolo.model = peft_backbone
  final_model = model.to(device)
  
  return final_model

def infuse_args(model: TriheadSegmentationModel):
  current_args = model.yolo.model.args if isinstance(model.yolo.model.args, dict) else {}

  if 'overlap_mask' not in current_args:
    current_args['overlap_mask'] = True

  current_args['box'] = 7.5
  current_args['cls'] = 0.5
  current_args['dfl'] = 1.5
    
  model.yolo.model.args = SimpleNamespace(**current_args)

def get_model(model_type: AblationStudyType):
  if model_type == 'vanilla':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=False,
      include_normals=False,
      device=device
    )
  elif model_type == 'additional-depth':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=True,
      include_normals=False,
      device=device
    )
  elif model_type == 'additional-normal':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=False,
      include_normals=True,
      device=device
    )
  elif model_type == 'additional-both':
    model = TriheadSegmentationModel(
      yolo_pt_path=f'{MODEL_WEIGHT_DIR}/yolo26m-seg.pt',
      include_depth=True,
      include_normals=True,
      device=device
    )
  
  infuse_args(model)
  final_model = create_peft_model(model)
  loss_balancer = UncertaintyLossWeighting()
  
  return final_model, loss_balancer

In [5]:
spatial_aug, photometric_aug = get_augmentation_pipeline()

def get_dataset_and_loader(model_type: AblationStudyType):
  if model_type == 'vanilla':
    include_depth = False
    include_normals = False
  elif model_type == 'additional-depth':
    include_depth = True
    include_normals = False
  elif model_type == 'additional-normal':
    include_depth = False
    include_normals = True
  elif model_type == 'additional-both':
    include_depth = True
    include_normals = True
  
  train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_depth=include_depth, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
  val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_depth=include_depth, include_normals=include_normals)
  test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_depth=include_depth, include_normals=include_normals)
  
  train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
  val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
  test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
  
  return {
    'train': (train_dataset, train_loader),
    'val': (val_dataset, val_loader),
    'test': (test_dataset, test_loader),
  }

In [6]:
def get_depth_and_normals_inclusion(model_type: AblationStudyType) -> tuple[bool, bool]:
  if model_type == 'vanilla':
    return False, False
  elif model_type == 'additional-depth':
    return True, False
  elif model_type == 'additional-normal':
    return False, True
  elif model_type == 'additional-both':
    return True, True

In [8]:
TOTAL_EPOCHS = 600

In [ ]:
try:
  for model_type in [
    'vanilla',
    'additional-depth',
    'additional-normal',
    'additional-both'
  ]:
    final_model, loss_balancer = get_model(model_type)
    dataset_and_loader = get_dataset_and_loader(model_type)
    include_depth, include_normals = get_depth_and_normals_inclusion(model_type)
    
    trainable_params = [p for p in final_model.parameters() if p.requires_grad]
    trainable_params.extend(loss_balancer.parameters())

    optimizer = AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)
    scaler = GradScaler('cuda')
    
    raw_yolo_architecture = final_model.yolo.model.base_model.model
    seg_loss_criterion = E2ELoss(raw_yolo_architecture, loss_fn=v8SegmentationLoss)
    depth_loss_criterion = L1Loss()
    
    early_stopping = EarlyStoppingAndCheckpointing()

    train_loader = dataset_and_loader['train'][1]
    val_loader = dataset_and_loader['val'][1]
    
    epoch_history = []
    train_loss_history = []
    val_loss_history = []

    for epoch in range(TOTAL_EPOCHS):
      epoch_train_loss = 0.0 
      epoch_train_seg_loss = 0.0
      epoch_weighted_train_seg_loss = 0.0
      epoch_train_depth_loss = 0.0
      epoch_weighted_train_depth_loss = 0.0
      epoch_train_normal_loss = 0.0
      epoch_weighted_train_normal_loss = 0.0
      
      for batch_idx, (batch_images, batch_targets) in enumerate(train_loader):
        optimizer.zero_grad()
        
        segmentation_out, depth_out, normal_out = final_model(batch_images.to(device=device))
        
        true_segmentation_map = batch_targets[0]
        true_depth_map = batch_targets[1]['depth'].to(device=device) if include_depth else None
        true_surface_normals = batch_targets[2]['normals'].to(device=device) if include_normals else None
        
        seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
        depth_loss = depth_loss_criterion(depth_out, true_depth_map) if include_depth else torch.tensor(0.0, device=device)
        normal_loss = compute_normal_loss(normal_out, true_surface_normals) if include_normals else torch.tensor(0.0, device=device)
        
        weighted_seg_loss = torch.abs(loss_balancer.alpha).to(device) * seg_loss
        weighted_depth_loss = torch.abs(loss_balancer.beta).to(device) * depth_loss
        weighted_normal_loss = torch.abs(loss_balancer.gamma).to(device) * normal_loss
        
        loss_total = weighted_seg_loss + weighted_depth_loss + weighted_normal_loss
        loss_total = loss_total.mean()
        loss_total.backward()
        
        optimizer.step()
        epoch_train_loss += loss_total.item()
        epoch_train_seg_loss += seg_loss.item()
        epoch_weighted_train_seg_loss = weighted_seg_loss.item()
        epoch_train_depth_loss += depth_loss.item()
        epoch_weighted_train_depth_loss = weighted_depth_loss.item()
        epoch_train_normal_loss += normal_loss.item()
        epoch_weighted_train_normal_loss += weighted_normal_loss.item()
        
        print(
          f"Epoch [{epoch:03d}/{TOTAL_EPOCHS:03d}] Batch [{batch_idx:04d}/{len(train_loader):04d}] | "
          f"Batch Train Loss: {loss_total.item():.4f} │ "
          f"LR: {optimizer.param_groups[0]['lr']:.2e}", end='\r'
        )
        
      avg_train_loss = epoch_train_loss / len(train_loader)
      avg_train_seg_loss = epoch_train_seg_loss / len(train_loader)
      avg_weighted_train_seg_loss = epoch_weighted_train_depth_loss / len(train_loader)
      avg_train_depth_loss = epoch_train_seg_loss / len(train_loader)
      avg_weighted_train_depth_loss = epoch_weighted_train_depth_loss / len(train_loader)
      avg_train_normal_loss = epoch_train_normal_loss / len(train_loader)
      avg_weighted_train_normal_loss = epoch_weighted_train_normal_loss / len(train_loader)

      epoch_val_loss = 0.0
      epoch_val_seg_loss = 0.0
      epoch_weighted_val_seg_loss = 0.0
      epoch_val_depth_loss = 0.0
      epoch_weighted_val_depth_loss = 0.0
      epoch_val_normal_loss = 0.0
      epoch_weight_val_normal_loss = 0.0
      
      with torch.no_grad():
        for batch_images_val, batch_targets_val in val_loader:
          if include_depth and include_normals:
            segmentation_out, depth_out, normal_out = final_model(batch_images_val.to(device=device))
            depth_out, normal_out = depth_out.to(device=device), normal_out.to(device=device)
          elif include_depth:
            segmentation_out, depth_out = final_model(batch_images_val.to(device=device))
            depth_out = depth_out.to(device=device)
          elif include_normals:
            segmentation_out, normal_out = final_model(batch_images_val.to(device=device))
            normal_out = normal_out.to(device=device)
          else:
            segmentation_out = final_model(batch_images_val.to(device=device))

          true_segmentation_map = batch_targets_val[0]
          true_depth_map = batch_targets_val[1]['depth'].to(device=device) if include_depth else None
          true_surface_normals = batch_targets_val[2]['normals'].to(device=device) if include_normals else None

          seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
          depth_loss = depth_loss_criterion(depth_out, true_depth_map) if include_depth else torch.tensor(0.0, device=device)
          normal_loss = compute_normal_loss(normal_out, true_surface_normals) if include_normals else torch.tensor(0.0, device=device)

          weighted_seg_loss = torch.abs(loss_balancer.alpha).to(device) * seg_loss
          weighted_depth_loss = torch.abs(loss_balancer.beta).to(device) * depth_loss
          weighted_normal_loss = torch.abs(loss_balancer.gamma).to(device) * normal_loss
      
          val_loss_total = weighted_seg_loss + weighted_depth_loss + weighted_normal_loss
          
          epoch_val_loss += val_loss_total.mean().item()
          epoch_val_seg_loss += seg_loss.item()
          epoch_weighted_val_seg_loss = weighted_seg_loss.item()
          epoch_val_depth_loss += depth_loss.item()
          epoch_weighted_val_depth_loss = weighted_depth_loss.item()
          epoch_val_normal_loss += normal_loss.item()
          epoch_weighted_val_normal_loss += weighted_normal_loss.item()
      
      avg_val_loss = epoch_val_loss / len(val_loader)
      avg_val_seg_loss = epoch_val_seg_loss / len(val_loader)
      avg_weighted_val_seg_loss = epoch_weighted_train_depth_loss / len(val_loader)
      avg_val_depth_loss = epoch_train_seg_loss / len(val_loader)
      avg_weighted_val_depth_loss = epoch_weighted_train_depth_loss / len(val_loader)
      avg_val_normal_loss = epoch_train_normal_loss / len(val_loader)
      avg_weighted_val_normal_loss = epoch_weighted_train_normal_loss / len(val_loader)
      
      # Record the metrics
      epoch_history.append(epoch)
      train_loss_history.append({
        'loss': avg_train_loss,
        'seg_loss': avg_train_seg_loss,
        'weighted_seg_loss': avg_weighted_train_seg_loss,
        'depth_loss': avg_train_depth_loss,
        'weighted_depth_loss': avg_weighted_train_depth_loss,
        'normal_loss': avg_train_normal_loss,
        'weighted_normal_loss': avg_weighted_train_normal_loss
      })
      val_loss_history.append({
        'loss': avg_val_loss,
        'seg_loss': avg_val_seg_loss,
        'weighted_seg_loss': avg_weighted_val_seg_loss,
        'depth_loss': avg_val_depth_loss,
        'weighted_depth_loss': avg_weighted_val_depth_loss,
        'normal_loss': avg_val_normal_loss,
        'weighted_normal_loss': avg_weighted_val_normal_loss
      })
      
      # Early Stopping evaluated ONCE per epoch using the average validation loss
      halt = early_stopping.record_and_check_if_halt(avg_val_loss, final_model.state_dict())
      
      if early_stopping.is_checkpoint():
        # TODO: Evaluate
        pass
      
      print(
        f"Epoch [{epoch:03d}/{TOTAL_EPOCHS:03d}] Batch [{batch_idx:04d}/{len(train_loader):04d}] | "
        f"Total Loss: {loss_total.item():.4f} │ "
        f"Seg: {seg_loss.item():.4f} │ "
        f"Depth: {depth_loss.item():.4f} │ "
        f"Norm: {normal_loss.item():.4f} │ "
        f"LR: {optimizer.param_groups[0]['lr']:.2e}", end='\r'
      )
      
      if halt:
        break
      
    early_stopping.save_weights()
    
    csv_filename = f"{model_type}_loss_history.csv"
    with open(csv_filename, mode='w', newline='') as file:
      writer = csv.writer(file)
      writer.writerow(['epoch', 'train_loss', 'val_loss'])
      for i in range(len(epoch_history)):
        writer.writerow([epoch_history[i], train_loss_history[i], val_loss_history[i]])
    print(f"Saved training history to {csv_filename}")
except Exception as exc:
  raise exc
finally:
  # Model & Optimization variables
  if 'peft_model' in locals(): del final_model
  if 'loss_balancer' in locals(): del loss_balancer
  if 'optimizer' in locals(): del optimizer
  if 'trainable_params' in locals(): del trainable_params
  if 'scaler' in locals(): del scaler
  
  # Data loaders
  if 'dataset_and_loader' in locals(): del dataset_and_loader
  if 'train_loader' in locals(): del train_loader
  if 'val_loader' in locals(): del val_loader
  
  # Trackers & Criteria
  if 'seg_loss_criterion' in locals(): del seg_loss_criterion
  if 'depth_loss_criterion' in locals(): del depth_loss_criterion
  if 'early_stopping' in locals(): del early_stopping
  if 'epoch_history' in locals(): del epoch_history
  if 'train_loss_history' in locals(): del train_loss_history
  if 'val_loss_history' in locals(): del val_loss_history
  
  # Train Loop Tensors
  if 'batch_images' in locals(): del batch_images
  if 'batch_targets' in locals(): del batch_targets
  if 'segmentation_out' in locals(): del segmentation_out
  if 'depth_out' in locals(): del depth_out
  if 'normal_out' in locals(): del normal_out
  
  if 'true_segmentation_map' in locals(): del true_segmentation_map
  if 'true_depth_map' in locals(): del true_depth_map
  if 'true_surface_normals' in locals(): del true_surface_normals
  
  if 'seg_loss' in locals(): del seg_loss
  if 'seg_loss_items' in locals(): del seg_loss_items
  if 'depth_loss' in locals(): del depth_loss
  if 'normal_loss' in locals(): del normal_loss
  if 'loss_total' in locals(): del loss_total
  
  # Val Loop Tensors
  if 'batch_images_val' in locals(): del batch_images_val
  if 'batch_targets_val' in locals(): del batch_targets_val
  if 'val_loss_total' in locals(): del val_loss_total
  
  # Force garbage collection and clear GPU cache
  import gc
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

trainable params: 1,368,496 || all params: 28,480,568 || trainable%: 4.8050


TypeError: 'bool' object is not callable

In [ ]:
# end_session()